# F14 en Colab — experimento corpus ZZU (SVM/RF)

Uso previsto: **VS Code local conectado al runtime de Colab por SSH**.

Qué necesita este notebook: solo el repo (los features `results/corpus/ventanas_features.csv`, 46 MB, están versionados). No requiere los 12 GB de `data/`.

## Conectar VS Code al runtime de Colab

Ejecutar primero las 2 celdas de «Setup SSH» de abajo. Luego, en VS Code:

1. Instalar la extensión **Remote - SSH** (ms-vscode-remote.remote-ssh).
2. Copiar a tu `~/.ssh/config` el host que imprime la celda (nombre Tailscale, puerto 22, user `root`).
3. `Remote-SSH: Connect to Host...` → ese host. Abrir la carpeta `/content/<repo>`.
4. En el terminal remoto: `pip install -e .` no hace falta (package=false); instalar dependencias con la celda «Entorno» de abajo o a mano.
5. `Select Interpreter` → el Python del Colab (`/usr/bin/python3`).

> Alternativa sin Tailscale: servicio `sshd` + túnel ngrok/Cloudflare (`colab-ssh`). Tailscale es más estable y no expone puertos públicos.

> Los runtimes de Colab se reinician (~12 h máximo, y al cerrar la pestaña): los resultados hay que empujarlos o copiarlos a Drive en la misma sesión (última celda).


## 1. Setup SSH (ejecutar en Colab antes de conectar VS Code)

In [ ]:
%%bash
apt-get -qq update >/dev/null && apt-get -qq install -y openssh-server >/dev/null
mkdir -p /run/sshd /root/.ssh && chmod 700 /root/.ssh
sed -i 's/#\?Port .*/Port 22/' /etc/ssh/sshd_config
nohup /usr/sbin/sshd >/dev/null 2>&1 &
echo "sshd arriba"

In [ ]:
# Pega tu clave pública local (cat ~/.ssh/id_ed25519.pub) y tu auth-key de Tailscale
# (https://login.tailscale.com/admin/settings/keys — key auth desechable, ephemeral+reusable)
PUBKEY = "ssh-ed25519 AAAA... tu@pc"
TS_AUTHKEY = "tskey-auth-..."

open("/root/.ssh/authorized_keys", "w").write(PUBKEY + "\n")
import os; os.chmod("/root/.ssh/authorized_keys", 0o600)
!curl -fsSL https://tailscale.com/install.sh | sh
!tailscale up --hostname colab-f14 --authkey="$TS_AUTHKEY" --ssh=false
!tailscale ip -4

Con la IP/nombre de Tailscale que imprime la celda, agrega a tu `~/.ssh/config`:

    Host colab-f14
        HostName <ip-tailscale>
        User root
        IdentityFile ~/.ssh/id_ed25519

y conecta con VS Code Remote-SSH. Las siguientes celdas también sirven si prefieres usar la UI de Colab directamente.

## 2. Repo + entorno

In [ ]:
REPO = "https://codeberg.org/norlek/Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG.git"
# Si el repo es privado: REPO = "https://<usuario>:<token-codeberg>@codeberg.org/norlek/....git"
%cd /content
!test -d Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG || git clone --depth 1 "$REPO"
%cd Software_Deteccion_Arritmias-Pediatricas-mediante-PDS_ECG
!git log --oneline -1
!ls -lh results/corpus/ventanas_features.csv

In [ ]:
# Dependencias (Colab trae numpy/scipy/pandas/sklearn; faltan wfdb y pywavelets).
# Se fija numpy<2 solo si sklearn/wfdb lo pidieran; Colab actual es compatible.
!pip install -q wfdb pywavelets matplotlib
import wfdb, pywt, sklearn, pandas
print("wfdb", wfdb.__version__, "| pywt", pywt.__version__, "| sklearn", sklearn.__version__)

## 3. F14 — experimento completo

In [ ]:
# Corre GroupKFold por paciente (SVM+RF) sobre el corpus ZZU-only.
# En el runtime estándar de Colab (2 vCPU) tarda como en local; el experimento no usa GPU.
!python scripts/ejecutar_experimento_corpus.py
!tail -5 results/experimento_corpus/kpi.json 2>/dev/null || true
!cat results/experimento_corpus/kpi.json

## 4. Rescatar resultados (no persisten al reciclar el runtime)

In [ ]:
# Opción A: push a Codeberg en la rama actual (recomendado: queda versionado)
!git config user.email "colab@runtime" && git config user.name "colab"
!git checkout -b resultados-colab
!git add results/experimento_corpus
!git commit -m "F14 re-ejecutado en Colab" && git push origin resultados-colab

# Opción B: copia a Google Drive (montar antes)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/arritmias_results
!cp -r results/experimento_corpus /content/drive/MyDrive/arritmias_results/

## Anexo — si hay que re-extraer features desde el dato crudo (no usual)

Solo si se cambia la definición de features (F13) o el índice de ventanas. Los 12 GB no están en git:

- **ZZU pECG** (público, CC BY 4.0, multipartito ~4 GB): URLs y MD5 en `docs/metodologia_zzu_pecg.md` §1 — `https://ndownloader.figshare.com/files/53524583` (.z01) y `.../53523902` (.zip). Descarga reanudable `curl -L -C -`, verificación MD5 contra el manifiesto (`https://api.figshare.com/v2/articles/27078763`), extracción 7z del zip multipartito.
- Luego: `python scripts/validar_corpus.py` → `extraer_features_corpus.py` → `ejecutar_experimento_corpus.py` (cadena F11→F14, commits de la rama `etapa2-corpus-combinado`).

El runtime estándar tiene ~100 GB de disco: alcanza para ZZU + procesamiento.